# Le pont entre l'activité et les prix s'est affaissé · *The bridge between activity and prices has sagged*

Notebook compagnon du chapitre **15. Croissance potentielle et output gap : la carte que personne ne peut mesurer** — [lire l'article](https://nmlab.io/ressources/croissance-potentielle-et-output-gap).
Companion notebook to chapter **15. Potential Growth and the Output Gap: The Map No One Can Measure** — [read the article](https://nmlab.io/en/ressources/potential-growth-and-output-gap).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# données FRED chargées dans build_figure


from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import pandas as pd

C, W = nm.COLORS, nm.WIDTH_PX


def wrap(ax, x: float, y: float, text: str, *, size: float = 19, color: str | None = None,
         weight: int = 500, ha: str = "left", va: str = "top", width: int = 42,
         lh: float = 1.5) -> int:
    """Écrit un texte replié à ``width`` caractères (coordonnées pixels)."""
    import textwrap
    lines: list[str] = []
    for para in text.split("\n"):
        lines += textwrap.wrap(para, width) or [""]
    ax.text(x, y, "\n".join(lines), fontsize=size, color=color or C["muted"],
            fontweight=weight, ha=ha, va=va, linespacing=lh, zorder=5)
    return len(lines)


def start(height: int = 1010) -> Figure:
    """Figure NMLab au format du site : 1747 px de large, fond sombre."""
    fig = nm.figure(height_px=height)
    fig.patch.set_facecolor(C["bg"])
    return fig


def dec(v: float, lang: str, n: int = 1, sign: bool = False) -> str:
    """Formate un nombre à la française (virgule, moins typographique) ou à l'anglaise."""
    s = f"{v:+.{n}f}" if sign else f"{v:.{n}f}"
    return s.replace("-", "−").replace(".", ",") if lang == "fr" else s


LABELS = {
    "fr": dict(
        title="Le pont entre l'activité et les prix s'est affaissé",
        sub="Écart de production et variation de l'inflation sous-jacente l'année suivante",
        xlab='Écart de production (% du potentiel)',
        ylab="Variation de l'inflation (points)",
        p1='1960-1985',
        p2='1986-2026',
        rl='corrélation',
        note="BEA, CBO et BLS (CPILFESL) via FRED ; calculs NMLab. Le même écart de production ne dit plus\nla même chose sur les prix : c'est l'aplatissement de la courbe de Phillips.",
    ),
    "en": dict(
        title='The bridge between activity and prices has sagged',
        sub='Output gap and the change in core inflation over the following year',
        xlab='Output gap (% of potential)',
        ylab='Change in inflation (points)',
        p1='1960-1985',
        p2='1986-2026',
        rl='correlation',
        note='BEA, CBO and BLS (CPILFESL) via FRED; NMLab calculations. The same output gap no longer says\nthe same thing about prices: this is the flattening of the Phillips curve.',
    ),
}


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab (libellés selon ``lang``)."""
    import numpy as np
    t = LABELS[lang]
    real, pot = nm.load_fred("GDPC1"), nm.load_fred("GDPPOT")
    core = nm.load_fred("CPILFESL").resample("QS").mean()
    infl = core.pct_change(4) * 100
    gap = ((real / pot - 1) * 100).dropna()
    d = pd.DataFrame({"g": gap, "i": infl, "i4": infl.shift(-4)}).dropna()
    d = d[d.index <= "2026-12-31"]; d["acc"] = d["i4"] - d["i"]
    dec = "," if lang == "fr" else "."
    fig = start(1010)
    nm.header(fig, t["title"], t["sub"])
    for i, (lo, hi, key, col) in enumerate(((1960, 1985, "p1", C["amber"]),
                                            (1986, 2026, "p2", C["blue"]))):
        ax = fig.add_axes([0.075 + i * 0.478, 0.205, 0.40, 0.50])
        x = d[(d.index.year >= lo) & (d.index.year <= hi)]
        r = x["g"].corr(x["acc"])
        ax.scatter(x["g"], x["acc"], s=46, color=col, alpha=0.6, edgecolors="none", zorder=3)
        b, a = np.polyfit(x["g"], x["acc"], 1)
        xs = np.array([x["g"].min(), x["g"].max()])
        ax.plot(xs, a + b * xs, color=C["rose"], lw=3.2, zorder=4)
        ax.axvline(0, color=C["edge"], lw=1.4); ax.axhline(0, color=C["edge"], lw=1.4)
        ax.set_title(f"{t[key]}   ·   {t['rl']} {r:+.2f}".replace(".", dec),
                     fontsize=22, fontweight=700, color=col, pad=18, loc="left")
        ax.set_xlabel(t["xlab"], fontsize=17.5, color=C["muted"], labelpad=12)
        if i == 0:
            ax.set_ylabel(t["ylab"], fontsize=17.5, color=C["muted"], labelpad=12)
        ax.set_xlim(-10, 7); ax.set_ylim(-7.5, 7.5)
        ax.grid(color=C["grid"], lw=1.1); ax.set_axisbelow(True)
        ax.tick_params(labelsize=16.5, colors=C["muted"], length=0)
        for sp in ("top", "right", "bottom", "left"): ax.spines[sp].set_visible(False)
    nm.footer(fig, t["note"])
    return fig


fig = build_figure(LANG)